# Image inference

This CPU example converts an RGB HWC image to model predictions and labelled
instance/type masks. By default it uses synthetic pixels and random weights for
an offline API check: the resulting masks have no biological meaning.

For real inference, set `image_path` and `weights_path` below to your RGB image
and local CellPose checkpoint. Use its documented encoder, class mapping and
preprocessing. The HGSC CellPose checkpoint uses global min-max normalization and
seven class channels; numeric class IDs are not verified semantic class names.
Checkpoint downloads are an explicit separate step, not part of this notebook.


In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from cellseg_models_pytorch.models.cellpose import CellPose
from cellseg_models_pytorch.inference.post_processor import PostProcessor
from cellseg_models_pytorch.transforms.functional.normalization import minmax_normalize

image_path = None  # e.g. Path("data/image.png")
weights_path = None  # e.g. Path("checkpoints/cellpose.safetensors")
device = torch.device("cpu")
torch.manual_seed(0)
torch.set_num_threads(2)

if image_path is None:
    image = np.random.default_rng(0).integers(0, 256, (64, 64, 3), dtype=np.uint8)
else:
    with Image.open(image_path) as source:
        image = np.asarray(source.convert("RGB"))
if any(size % 32 != 0 for size in image.shape[:2]):
    raise ValueError("Use image dimensions divisible by 32.")
image = minmax_normalize(image, copy=True)

In [ ]:
if weights_path is None:
    model = CellPose(3, enc_name="resnet18", enc_pretrain=False, device=device)
else:
    model = CellPose.from_pretrained(Path(weights_path), device=device)
model.set_inference_mode(mixed_precision=False)
model.post_processor = PostProcessor("cellpose", {"use_gpu": False})
prediction = model.predict(image)
result = model.post_process(prediction, use_async_postproc=False)
instances, classes = result["nuc"][0]
print("Instance count:", len(np.unique(instances[instances > 0])))

`predict` accepts HWC NumPy/PIL images or BCHW tensors; normalization is your
responsibility. Dense outputs precede model-specific reconstruction. `nuc` contains
one `(instance_mask, type_mask)` pair per image. `cyto` and `tissue` are `None`
for this nuclei-only model. In the instance mask, 0 is background and each positive
number identifies one object. The type mask gives each object a class number;
class 0 can also be assigned to an object.

For larger images, `predict(..., use_sliding_win=True, window_size=(256, 256),
stride=128)` uses tiled prediction. Window and stride choices must fit the encoder;
this notebook does not validate whole-slide reading, merging or GPU execution.


In [ ]:
# Preserve labels without lossy image conversion; do not overwrite an existing result.
output_path = Path("segmentation.npz")
# Uncomment to save after choosing a destination:
# with output_path.open("xb") as output:
#     np.savez_compressed(output, instances=instances, classes=classes)